# Model Monitoring Drift Checks

## Objective

Build an empirical monitoring snapshot from the held-out processed test split. The notebook uses the selected model and the shared `yield_risk.monitoring.build_monitoring_summary` utility to compare deterministic reference and current batches for missingness drift, feature distribution drift, prediction score drift, and high-risk-rate drift.

In [1]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    """Find the nearest parent directory containing src/."""
    for candidate in (start, *start.parents):
        if (candidate / "src").is_dir():
            return candidate
    raise RuntimeError("Could not locate repository root containing src/.")


try:
    start_dir = Path(__file__).resolve().parent
except NameError:
    start_dir = Path.cwd().resolve()

repo_root = find_repo_root(start_dir)
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

repo_root

WindowsPath('C:/Users/Virgil/Code/semiconductor-yield-root-cause')

In [2]:
import joblib
import pandas as pd

from yield_risk.config import load_config
from yield_risk.monitoring import build_monitoring_summary

## Load Artifacts

Load the run configuration, processed held-out test data, selected model, and model comparison table from repository-root-relative paths.

In [3]:
cfg = load_config(repo_root / "configs" / "config.yaml")

test = pd.read_csv(repo_root / cfg.paths.splits_dir / "test.csv")
sensor_cols = [column for column in test.columns if column.startswith("sensor_")]
model = joblib.load(repo_root / cfg.paths.models_dir / "selected_model.joblib")
model_comparison = pd.read_csv(repo_root / cfg.paths.reports_dir / "model_comparison.csv")

selected = model_comparison.loc[model_comparison["selected"]].iloc[0]
threshold = float(selected["frozen_threshold"])

artifact_summary = pd.DataFrame(
    [
        {
            "test_rows": len(test),
            "sensor_features": len(sensor_cols),
            "selected_model": selected["model"],
            "operating_threshold": threshold,
        }
    ]
)
artifact_summary

,test_rows,sensor_features,selected_model,operating_threshold
0,236,590,random_forest,0.14


In [4]:
model_comparison

,model,cv_pr_auc_mean,test_pr_auc,test_roc_auc,test_recall,test_precision,frozen_threshold,expected_cost,selected
0,dummy,0.065982,0.068367,0.503977,0.0625,0.076923,0.01,162.0,False
1,logistic_regression,0.179721,0.168193,0.725568,0.3125,0.142857,0.52,140.0,False
2,random_forest,0.226585,0.221624,0.792614,0.8125,0.196970,0.14,83.0,True
3,xgboost,0.206185,0.213262,0.785795,0.6250,0.156250,0.02,114.0,False


## Deterministic Monitoring Batches

Split the held-out test data into fixed reference and current halves, then score both batches with the selected model.

In [5]:
midpoint = len(test) // 2
reference = test.iloc[:midpoint].copy()
current = test.iloc[midpoint:].copy()

reference_scores = model.predict_proba(reference[sensor_cols])[:, 1]
current_scores = model.predict_proba(current[sensor_cols])[:, 1]

pd.DataFrame(
    [
        {
            "reference_rows": len(reference),
            "current_rows": len(current),
            "reference_score_mean": reference_scores.mean(),
            "current_score_mean": current_scores.mean(),
        }
    ]
)

,reference_rows,current_rows,reference_score_mean,current_score_mean
0,118,118,0.11894,0.113882


## Monitoring Summary

Build all monitoring sections through the shared library utility.

In [6]:
summary = build_monitoring_summary(
    reference,
    current,
    sensor_cols,
    reference_scores,
    current_scores,
    high_risk_threshold=threshold,
)
summary

MonitoringSummary(missingness=MissingnessDriftSummary(results=[MissingnessDriftResult(feature='sensor_000', reference_missing_rate=0.0, current_missing_rate=0.0, missing_rate_delta=0.0, alert=False), MissingnessDriftResult(feature='sensor_001', reference_missing_rate=0.0, current_missing_rate=0.00847457627118644, missing_rate_delta=0.00847457627118644, alert=False), MissingnessDriftResult(feature='sensor_002', reference_missing_rate=0.0, current_missing_rate=0.0, missing_rate_delta=0.0, alert=False), MissingnessDriftResult(feature='sensor_003', reference_missing_rate=0.0, current_missing_rate=0.0, missing_rate_delta=0.0, alert=False), MissingnessDriftResult(feature='sensor_004', reference_missing_rate=0.0, current_missing_rate=0.0, missing_rate_delta=0.0, alert=False), MissingnessDriftResult(feature='sensor_005', reference_missing_rate=0.0, current_missing_rate=0.0, missing_rate_delta=0.0, alert=False), MissingnessDriftResult(feature='sensor_006', reference_missing_rate=0.0, current_mi

### Missingness Drift Alerts

In [7]:
missingness_alerts = summary.missingness.to_frame().query("alert")
missingness_alerts.head(10)

,feature,reference_missing_rate,current_missing_rate,missing_rate_delta,alert
109,sensor_109,0.635593,0.559322,0.076271,True
110,sensor_110,0.635593,0.559322,0.076271,True
111,sensor_111,0.635593,0.559322,0.076271,True
157,sensor_157,0.949153,0.889831,0.059322,True
158,sensor_158,0.949153,0.889831,0.059322,True
244,sensor_244,0.635593,0.559322,0.076271,True
245,sensor_245,0.635593,0.559322,0.076271,True
246,sensor_246,0.635593,0.559322,0.076271,True
292,sensor_292,0.949153,0.889831,0.059322,True
293,sensor_293,0.949153,0.889831,0.059322,True


### Top Feature Drift Rows

In [8]:
summary.features.to_frame().sort_values(
    ["alert", "mean_delta", "ks_statistic"], ascending=[False, False, False]
).head(10)

,feature,reference_mean,current_mean,mean_delta,reference_std,current_std,std_delta,reference_median,current_median,median_delta,ks_statistic,ks_p_value,alert
162,sensor_162,5184.358974,4476.771186,707.587788,7099.451839,6541.249280,558.202559,1440.00000,1678.50000,238.50000,0.096407,0.608627,True
297,sensor_297,2525.560374,2164.861387,360.698986,3378.288467,3171.796360,206.492107,692.42500,764.12355,71.69855,0.096480,0.607691,True
161,sensor_161,3795.863248,4148.008475,352.145227,3553.957858,4615.587756,1061.629897,2668.00000,2475.50000,192.50000,0.065769,0.945819,True
140,sensor_140,169.760179,0.335056,169.425123,1290.503897,0.370674,1290.133223,0.23020,0.21855,0.01165,0.067797,0.932067,True
296,sensor_296,1758.097676,1913.190998,155.093322,1726.980265,2227.528592,500.548327,1157.77270,1169.95860,12.18590,0.049761,0.997069,True
158,sensor_158,1135.450367,1000.892277,134.558090,158.058607,191.919304,33.860697,1118.90040,1032.19970,86.70070,0.346154,0.618264,True
24,sensor_024,-181.647436,-50.223871,131.423565,2708.542634,3238.579785,530.037151,87.00000,-153.50000,240.50000,0.106258,0.485342,True
3,sensor_003,1442.978016,1375.068383,67.909633,471.950252,440.142547,31.807705,1418.90860,1229.75410,189.15450,0.161017,0.083707,True
275,sensor_275,56.577118,0.113751,56.463367,430.092173,0.155912,429.936261,0.07535,0.07075,0.00460,0.076271,0.856477,True
499,sensor_499,240.387507,287.071596,46.684089,322.221483,338.879189,16.657705,0.00000,0.00000,0.00000,0.101695,0.541239,True


### Prediction Drift Summary

In [9]:
pd.DataFrame([summary.predictions.__dict__])

,reference_mean,current_mean,mean_delta,reference_median,current_median,median_delta,reference_std,current_std,std_delta,reference_p90,current_p90,p90_delta,ks_statistic,ks_p_value,alert
0,0.11894,0.113882,0.005059,0.109303,0.107855,0.001448,0.057991,0.058257,0.000266,0.194084,0.201875,0.007791,0.09322,0.649972,False


### High-Risk-Rate Drift Summary

In [10]:
pd.DataFrame([summary.high_risk_rate.__dict__])

,threshold,reference_rate,current_rate,rate_delta,rate_delta_threshold,alert
0,0.14,0.271186,0.288136,0.016949,0.05,False


## Monitoring Interpretation

This notebook uses historical SECOM data and anonymous sensors. A cluster of high-missingness sensors trips the missingness alert as their missing rate shifts between the two halves, and feature drift is broad under the raw thresholds (most sensors flag at least one moving statistic). The prediction-score and high-risk-rate checks, by contrast, stay quiet: the selected model's score distribution and its high-risk rate at the 0.14 operating threshold are stable across the reference and current batches. Because the feature-level alerts are broad while the model-output checks do not fire, the next step is calibrating monitoring thresholds against an operating policy before wiring the checks into a production feedback loop.